Імпорти

In [7]:
import sys
import time
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    TEXT_COLUMN,
    TARGET_COLUMN,
)

from src.pipelines import (
    build_logistic_pipeline,
    build_svm_pipeline,
)

from src.evaluation import (
    calculate_classification_metrics,
    create_results_table,
)

1. Завантажуємо дані

In [8]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df[TEXT_COLUMN]
y_train = train_df[TARGET_COLUMN]

X_val = val_df[TEXT_COLUMN]
y_val = val_df[TARGET_COLUMN]

X_test = test_df[TEXT_COLUMN]
y_test = test_df[TARGET_COLUMN]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (7999,)
Validation: (2000,)
Test: (3080,)


2. Робимо Dummy baseline

In [10]:
results = []

dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42,
)

start_time = time.perf_counter()

dummy_model.fit(X_train, y_train)

dummy_train_time = time.perf_counter() - start_time
dummy_val_pred = dummy_model.predict(X_val)

dummy_result = calculate_classification_metrics(
    model_name="DummyClassifier",
    y_true=y_val,
    y_pred=dummy_val_pred,
    train_time=dummy_train_time,
    parameters={
        "strategy": "most_frequent",
    },
)

results.append(dummy_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec
0,DummyClassifier,{'strategy': 'most_frequent'},0.019,0.000484,0.000709,0.000247,0.012987,0.002914


Як бачимо з результатів, DummyClassifier показує дуже низькі результати, оскільки він завжди передбачає найчастіший клас. Це підтверджує, що завдання не може бути вирішене за допомогою наївних стратегій і вимагає семантичного розуміння запитів клієнтів.

3. Тренуємо Logistic Regression

In [11]:
logistic_pipeline = build_logistic_pipeline()

In [12]:
start_time = time.perf_counter()

logistic_pipeline.fit(X_train, y_train)

logistic_train_time = time.perf_counter() - start_time
logistic_val_pred = logistic_pipeline.predict(X_val)

In [13]:
logistic_result = calculate_classification_metrics(
    model_name="TF-IDF + Logistic Regression",
    y_true=y_val,
    y_pred=logistic_val_pred,
    train_time=logistic_train_time,
    parameters={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "C": 1.0,
    },
)

results.append(logistic_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec
0,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.841,0.829135,0.839451,0.854706,0.824639,2.992030
1,DummyClassifier,{'strategy': 'most_frequent'},0.019,0.000484,0.000709,0.000247,0.012987,0.002914


In [14]:
tfidf = logistic_pipeline.named_steps["tfidf"]

print(
    "Number of TF-IDF features:",
    len(tfidf.get_feature_names_out()),
)

Number of TF-IDF features: 8913


In [15]:
logistic_result["n_features"] = len(
    tfidf.get_feature_names_out()
)

In [16]:
label_mapping = (
    train_df[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["label_text"]
    .to_dict()
)

target_names = [
    label_mapping[label]
    for label in sorted(label_mapping)
]

In [17]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val,
        logistic_val_pred,
        labels=sorted(label_mapping),
        target_names=target_names,
        zero_division=0,
    )
)

                                                  precision    recall  f1-score   support

                                activate_my_card       0.97      0.91      0.94        32
                                       age_limit       0.95      0.95      0.95        22
                         apple_pay_or_google_pay       1.00      1.00      1.00        25
                                     atm_support       0.83      0.88      0.86        17
                                automatic_top_up       1.00      0.84      0.91        25
         balance_not_updated_after_bank_transfer       0.75      0.79      0.77        34
balance_not_updated_after_cheque_or_cash_deposit       0.86      1.00      0.92        36
                         beneficiary_not_allowed       0.92      0.77      0.84        31
                                 cancel_transfer       0.93      0.90      0.92        31
                            card_about_to_expire       0.80      0.92      0.86        26
         

4. Робимо Linear SVM

In [18]:
svm_pipeline = build_svm_pipeline()

In [19]:
start_time = time.perf_counter()

svm_pipeline.fit(X_train, y_train)

svm_train_time = time.perf_counter() - start_time
svm_val_pred = svm_pipeline.predict(X_val)

In [20]:
svm_result = calculate_classification_metrics(
    model_name="TF-IDF + Linear SVM",
    y_true=y_val,
    y_pred=svm_val_pred,
    train_time=svm_train_time,
    parameters={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "C": 1.0,
    },
)

svm_tfidf = svm_pipeline.named_steps["tfidf"]

svm_result["n_features"] = len(
    svm_tfidf.get_feature_names_out()
)

results.append(svm_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,n_features
0,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.370692,8913.0
1,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.992030,8913.0
2,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.002914,NaN


5. Порівняємо результати

In [21]:
results_df = create_results_table(results)

results_df[
    [
        "model",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "macro_precision",
        "macro_recall",
        "train_time_sec",
        "n_features",
    ]
]

,model,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,n_features
0,TF-IDF + Linear SVM,0.8755,0.870909,0.875092,0.879338,0.868093,0.370692,8913.0
1,TF-IDF + Logistic Regression,0.8410,0.829135,0.839451,0.854706,0.824639,2.992030,8913.0
2,DummyClassifier,0.0190,0.000484,0.000709,0.000247,0.012987,0.002914,NaN


Як бачимо із порівняльної таблиці, моделі TF-IDF + Linier SVM та TF-IDF + Logistic Regression показуюють дуже хороші результати навіть без тюнінгу. Macro F1 Linear SVM тановить 0.87, а для Logistic Regression 0.83.

6. Робимо тюнінг Logistic Regression і Linear SVM

6.1 GridSearch для Logistic Regression

In [22]:
from sklearn.model_selection import GridSearchCV

logistic_pipeline = build_logistic_pipeline()

logistic_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0],
}

logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [26]:
import time
import warnings
warnings.filterwarnings('ignore')

start_time = time.perf_counter()

logistic_grid.fit(X_train, y_train)

logistic_grid_time = time.perf_counter() - start_time

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [27]:
print("Best parameters:")
print(logistic_grid.best_params_)

print("Best CV Macro F1:")
print(logistic_grid.best_score_)

Best parameters:
{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Best CV Macro F1:
0.8529892233070906


Оцінка на validation

In [35]:
logistic_tuned_val_pred = logistic_grid.predict(X_val)

logistic_tuned_result = calculate_classification_metrics(
    model_name="TF-IDF + Logistic Regression (tuned)",
    y_true=y_val,
    y_pred=logistic_tuned_val_pred,
    train_time=logistic_grid_time,
    parameters=logistic_grid.best_params_,
)

logistic_tuned_result["n_features"] = len(
    logistic_grid
    .best_estimator_
    .named_steps["tfidf"]
    .get_feature_names_out()
)

results.append(logistic_tuned_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,n_features
0,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.370692,8913.0
1,TF-IDF + Logistic Regression (tuned),"{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tf...",0.8675,0.862007,0.867527,0.874387,0.858843,14.432031,1331.0
2,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.992030,8913.0
3,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.002914,NaN


6.2. GridSearch для Linear SVM

In [36]:
svm_pipeline = build_svm_pipeline()

svm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0],
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [37]:
start_time = time.perf_counter()

svm_grid.fit(X_train, y_train)

svm_grid_time = time.perf_counter() - start_time

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [38]:
print("Best parameters:")
print(svm_grid.best_params_)

print("Best CV Macro F1:")
print(svm_grid.best_score_)

Best parameters:
{'classifier__C': 1.0, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Best CV Macro F1:
0.8638637953920215


In [39]:
svm_tuned_val_pred = svm_grid.predict(X_val)

svm_tuned_result = calculate_classification_metrics(
    model_name="TF-IDF + Linear SVM (tuned)",
    y_true=y_val,
    y_pred=svm_tuned_val_pred,
    train_time=svm_grid_time,
    parameters=svm_grid.best_params_,
)

svm_tuned_result["n_features"] = len(
    svm_grid
    .best_estimator_
    .named_steps["tfidf"]
    .get_feature_names_out()
)

results.append(svm_tuned_result)

results_df = create_results_table(results)
results_df

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,n_features
0,TF-IDF + Linear SVM (tuned),"{'classifier__C': 1.0, 'tfidf__min_df': 1, 'tf...",0.8790,0.873237,0.878779,0.881031,0.871352,4.722258,2158.0
1,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.370692,8913.0
2,TF-IDF + Logistic Regression (tuned),"{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tf...",0.8675,0.862007,0.867527,0.874387,0.858843,14.432031,1331.0
3,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.992030,8913.0
4,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.002914,NaN


7. Зберігаємо найкращі класичні моделі

In [40]:
import joblib

from src.config import MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    logistic_grid.best_estimator_,
    MODELS_DIR / "logistic_pipeline_tuned.joblib",
)

joblib.dump(
    svm_grid.best_estimator_,
    MODELS_DIR / "svm_pipeline_tuned.joblib",
)

['/Users/oleh/PycharmProjects/banking-intent-classification/models/svm_pipeline_tuned.joblib']